# 7.5 Backpropagation Without Handwaving

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook makes the gradient mechanics visible. The goal is to turn backpropagation from a phrase into an inspectable procedure.


## 7.5.1 Computation graph intuition

### Why It Matters
PyTorch records operations on tensors that require gradients. That graph is what lets autograd work backward from the loss.


In [ ]:
import torch

x = torch.tensor([2.0], requires_grad=True)
y = x ** 2 + 3 * x
z = y.mean()
z.backward()
{"y_value": float(y.item()), "dz_dx": float(x.grad.item())}


### Interpretation
The derivative comes from the chain of operations that produced `z`. Autograd is following that chain for you.


## 7.5.2 Chain rule in practice

### Why It Matters
The chain rule feels less abstract when you compare autograd with a hand-derived derivative on a tiny expression.


In [ ]:
import torch

x = torch.tensor([1.5], requires_grad=True)
y = (2 * x + 1) ** 3
y.backward()
manual = 3 * (2 * 1.5 + 1) ** 2 * 2
{"autograd": round(float(x.grad.item()), 3), "manual_derivative": round(float(manual), 3)}


### Interpretation
Matching values confirm that autograd is applying the same chain rule you would write on paper.


## 7.5.3 Autograd over network parameters

### Why It Matters
Backpropagation in neural networks means computing gradients not only with respect to inputs, but also with respect to each learnable weight and bias.


In [ ]:
import torch
from torch import nn

model = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4, 1))
X = torch.randn(5, 3)
y = torch.randn(5, 1)
loss = nn.MSELoss()(model(X), y)
loss.backward()
{name: round(float(param.grad.norm().item()), 4) for name, param in model.named_parameters()}


### Interpretation
Every parameter receives a gradient tensor shaped exactly like the parameter itself.


## 7.5.4 Why zeroing gradients matters

### Why It Matters
PyTorch accumulates gradients by default. That is useful for some advanced patterns, but it surprises beginners who expect each backward pass to start fresh.


In [ ]:
import torch

w = torch.tensor([1.0], requires_grad=True)
losses = []
for _ in range(2):
    loss = (w - 3).pow(2)
    loss.backward()
    losses.append(round(float(w.grad.item()), 3))
w.grad.zero_()
loss = (w - 3).pow(2)
loss.backward()
{"after_two_backward_calls_without_zero": losses, "after_zero_then_backward": round(float(w.grad.item()), 3)}


### Interpretation
The second gradient doubles because the first one was still stored. A clean training loop always handles this deliberately.


## 7.5.5 Vanishing and exploding intuition

### Why It Matters
Very deep compositions can shrink or amplify gradients depending on the activations and weights. This example compares repeated multiplication by small versus large factors.


In [ ]:
import torch

small = torch.tensor(1.0, requires_grad=True)
expr_small = small
for _ in range(12):
    expr_small = expr_small * 0.5
expr_small.backward()
grad_small = small.grad.item()

large = torch.tensor(1.0, requires_grad=True)
expr_large = large
for _ in range(12):
    expr_large = expr_large * 1.5
expr_large.backward()
grad_large = large.grad.item()

{"vanishing_style_gradient": round(float(grad_small), 8), "exploding_style_gradient": round(float(grad_large), 3)}


### Interpretation
Real networks are more complex than repeated scalar multiplication, but the intuition is the same: gradients can disappear or blow up across long chains.


## 7.5.6 Inspect gradients in a small network

### Why It Matters
You do not have to treat gradients as invisible. Inspecting them directly is often the fastest way to debug a bad training run.


In [ ]:
import torch
from torch import nn

torch.manual_seed(18)
X = torch.randn(30, 2)
y = ((X[:, 0] + X[:, 1]) > 0).float().unsqueeze(1)
model = nn.Sequential(nn.Linear(2, 6), nn.Tanh(), nn.Linear(6, 1), nn.Sigmoid())
loss = nn.BCELoss()(model(X), y)
loss.backward()

gradient_summary = {
    name: {
        "mean_abs_grad": round(float(param.grad.abs().mean().item()), 5),
        "max_abs_grad": round(float(param.grad.abs().max().item()), 5),
    }
    for name, param in model.named_parameters()
}
gradient_summary


### Interpretation
Gradient inspection helps answer whether the model is actually learning signal or getting stuck with tiny or unstable updates.
